# 04 — Training fundamentals, one update at a time

This is a four-parameter CPU teaching model. We predict whether total time is at most 30 minutes, using eight handwritten numeric examples: four for training and four held out for validation. This is a pedagogical table, not a recipe corpus or RecipeTriage-Bench-v1. We do not train the local language model or contact Fireworks.

Run the cells from top to bottom. The notebook contains all its Python code and **does not import `recipetriage_ml`**, avoiding package-path setup problems. Select a Python environment with PyTorch and an IPython kernel; the project setup is in TRAINING_FUNDAMENTALS.md. No model downloads are required after dependencies are installed.

Before running each cell, predict what will happen. At the end, explain the loop in your own words. We will discuss SFTTrainer only after that checkpoint.

In [1]:
from pathlib import Path
import csv
import math
import sys
import torch
from torch import nn
from IPython.display import Markdown, display

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "docker-compose.yml").is_file()), Path.cwd())
OUTPUT = ROOT / "training-results" / "notebook"
print("Results will be saved in:", OUTPUT)

Python: /Users/smohandoss/Documents/Codex/2026-09-09/files-mentioned-by-the-user-recipetriage/outputs/recipetriage/.venv/bin/python
PyTorch: 2.14.0
Results will be saved in: /Users/smohandoss/Documents/Codex/2026-09-09/files-mentioned-by-the-user-recipetriage/outputs/recipetriage/training-results/notebook


## 1. Labels and tensor shapes

Our class mapping is **0 = more than 30 minutes**, **1 = at most 30 minutes**. The label is the correct answer, not a model score. Cross-entropy uses integer class indices here.

| Split | Minutes | Labels |
|---|---|---|
| Train | 10, 20, 40, 60 | 1, 1, 0, 0 |
| Validation | 15, 25, 45, 55 | 1, 1, 0, 0 |

We scale each input with $x=(\text{minutes}-30)/30$. The rule is fixed before training. Validation examples never contribute to a backward pass. In a real application, known elapsed time could be classified with a simple rule; we deliberately learn it here to inspect optimization.

This is one class per example. RecipeTriage's actual multi-label targets need a different formulation, such as independent binary outputs or the existing structured text approach. Do not directly substitute this two-class loss for the seven-label task.

In [2]:
def tiny_data():
    """Handwritten teaching examples: class 0 = over 30 min; class 1 = <=30 min."""
    train_minutes = torch.tensor([[10.0], [20.0], [40.0], [60.0]])
    valid_minutes = torch.tensor([[15.0], [25.0], [45.0], [55.0]])
    train_labels = torch.tensor([1, 1, 0, 0], dtype=torch.long)
    valid_labels = torch.tensor([1, 1, 0, 0], dtype=torch.long)
    # A fixed, documented scaling rule; no statistics are learned from validation.
    return (train_minutes - 30) / 30, train_labels, (valid_minutes - 30) / 30, valid_labels

x_train, y_train, x_valid, y_valid = tiny_data()
print("Inputs:", x_train.tolist(), "shape:", tuple(x_train.shape))
print("Labels:", y_train.tolist(), "shape:", tuple(y_train.shape), "dtype:", y_train.dtype)

Inputs: [[-0.6666666865348816], [-0.3333333432674408], [0.3333333432674408], [1.0]] shape: (4, 1)
Labels: [1, 1, 0, 0] shape: (4,) dtype: torch.int64


## 2. Forward pass

The model has two weights and two biases. For a batch of $B$ examples, $X$ has shape $[B,1]$, $W$ is $[2,1]$, and $b$ is $[2]$:

$$z=XW^\top+b \quad\text{has shape }[B,2].$$

These raw scores are **logits**. For inspection, softmax converts them to probabilities:

$$p_k=\frac{\exp(z_k)}{\sum_j\exp(z_j)}.$$

We initialize this simple linear model to zero, so its first probabilities are $[0.5,0.5]$. Zero initialization is convenient for this calculation; it is not a suitable general initialization method for hidden layers in deep networks.

In [3]:
def make_model():
    # One input, two logits: two weights plus two biases = four trainable numbers.
    # Zero initialization is intentional for this linear lesson, not a deep-network recipe.
    with torch.random.fork_rng():
        model = nn.Linear(1, 2)
    nn.init.zeros_(model.weight)
    nn.init.zeros_(model.bias)
    return model

model = make_model()
model.train()
optimizer = torch.optim.SGD(model.parameters(), lr=0.2)
loss_fn = nn.CrossEntropyLoss()
x_one, y_one = x_train[:1], y_train[:1]  # 10 minutes, class 1.
logits = model(x_one)
probabilities = torch.softmax(logits, dim=1)
print("Trainable numbers:", sum(p.numel() for p in model.parameters()))
print("Logits:", logits.detach().tolist())
print("Probabilities:", probabilities.detach().tolist())

Trainable numbers: 4
Logits: [[0.0, 0.0]]
Probabilities: [[0.5, 0.5]]


## 3. Cross-entropy loss

For one example, $L=-\log p_y$. For a batch we average:

$$L_{\text{batch}}=-\frac1B\sum_{i=1}^B\log p_{i,y_i}.$$

At probability $0.5$, loss is $-\log 0.5\approx0.693147$. At $0.25$ it is $1.386294$; at $0.75$ it is $0.287682$. Confidently missing the correct class is costly.

Pass **raw logits** to `nn.CrossEntropyLoss`. It performs the stable log-softmax/loss calculation internally. Do not feed it the softmax probabilities shown for inspection. See the [PyTorch loss contract](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html).

In [4]:
loss = loss_fn(logits, y_one)
manual_loss = -torch.log(probabilities[0, y_one.item()])
torch.testing.assert_close(loss, manual_loss)
loss_before = loss.item()  # A Python number for logging, not for backward().
print("Cross-entropy:", loss_before)
print("Manual -log(p_correct):", manual_loss.item())

Cross-entropy: 0.6931471824645996
Manual -log(p_correct): 0.6931471824645996


## 4. Gradient and backpropagation

A gradient is a vector of derivatives: $\nabla_\theta L$ describes local sensitivity of loss to parameters. A positive derivative means increasing that parameter locally increases loss, holding the others fixed.

For one example with softmax cross-entropy:

$$\frac{\partial L}{\partial z_k}=p_k-\mathbf1[k=y],\quad
\frac{\partial L}{\partial W_k}=(p_k-\mathbf1[k=y])x,\quad
\frac{\partial L}{\partial b_k}=p_k-\mathbf1[k=y].$$

For this 10-minute example, $x=-2/3$, $p=[0.5,0.5]$, and $y=1$. Therefore the weight gradient is $[-1/3,+1/3]^\top$ and the bias gradient is $[+1/2,-1/2]$.

`loss.backward()` performs **backpropagation**: it follows the recorded computation graph backward and applies the chain rule. It computes gradients; it does not update weights.

The gradient norm $\|g\|_2=\sqrt{\sum_i g_i^2}$ summarizes magnitude. The norm loses the signs, so inspect the actual gradient tensors too. Very large, tiny, or non-finite norms can help diagnose problems, but there is no universal ideal norm.

In [5]:
def gradient_norms(model):
    # ||g||_2 = sqrt(sum_i g_i^2). None means no gradient tensor is stored.
    return {name: None if parameter.grad is None else parameter.grad.norm().item()
            for name, parameter in model.named_parameters()}

optimizer.zero_grad(set_to_none=True)
weights_before_backward = {name: p.detach().clone() for name, p in model.named_parameters()}
loss.backward()
for name, parameter in model.named_parameters():
    print(name, "gradient:", parameter.grad.tolist())
    torch.testing.assert_close(parameter, weights_before_backward[name])
print("Gradient norms:", gradient_norms(model))
torch.testing.assert_close(model.weight.grad, torch.tensor([[-1/3], [1/3]]))
torch.testing.assert_close(model.bias.grad, torch.tensor([0.5, -0.5]))

weight gradient: [[-0.3333333432674408], [0.3333333432674408]]
bias gradient: [0.5, -0.5]
Gradient norms: {'weight': 0.4714045524597168, 'bias': 0.7071067690849304}


## 5. Optimizer, learning rate, and one step

Plain stochastic gradient descent (SGD) makes the update:

$$\theta_{t+1}=\theta_t-\eta\nabla_\theta L_t.$$

$\eta$ is the **learning rate**. With $\eta=0.2$, the two new weights should be $[1/15,-1/15]^\top$ and biases $[-0.1,0.1]$.

We now evaluate loss again on the **same example**, using a new forward pass. The previous `loss` tensor keeps its old value. Calling `optimizer.step()` does not rewrite it. A loss decrease is expected for this small step, but is not guaranteed for every optimizer, batch, or learning rate.

In [6]:
gradients = {name: p.grad.clone() for name, p in model.named_parameters()}
optimizer.step()
for name, parameter in model.named_parameters():
    expected = weights_before_backward[name] - 0.2 * gradients[name]
    torch.testing.assert_close(parameter, expected)
    print(name, "after SGD:", parameter.detach().tolist())
with torch.no_grad():
    new_logits = model(x_one)
    loss_after = loss_fn(new_logits, y_one).item()
    new_probabilities = torch.softmax(new_logits, dim=1)
print(f"Same-example loss BEFORE: {loss_before:.6f}; AFTER: {loss_after:.6f}")
print("New probabilities:", new_probabilities.tolist())
assert loss_after < loss_before
print("Gradients still stored after step:", gradient_norms(model))

weight after SGD: [[0.06666667014360428], [-0.06666667014360428]]
bias after SGD: [-0.10000000149011612, 0.10000000149011612]
Same-example loss BEFORE: 0.693147; AFTER: 0.559099
New probabilities: [[0.4282759428024292, 0.5717241168022156]]
Gradients still stored after step: {'weight': 0.4714045524597168, 'bias': 0.7071067690849304}


## 6. Why zero_grad matters

PyTorch **adds** each backward pass into `.grad`. With unchanged parameters and the same examples, two forward/backward passes produce $g+g=2g$ unless you clear the buffers. The computational graph must be recomputed for the second backward pass; we do that below.

`optimizer.zero_grad(set_to_none=True)` makes each stored gradient `None`. `set_to_none=False` zeros existing gradient tensors instead. Neither operation resets weights. `optimizer.step()` does not clear gradients automatically. Intentional gradient accumulation is possible, but this lesson takes one update per batch. [PyTorch zero_grad](https://docs.pytorch.org/docs/main/generated/torch.optim.Optimizer.zero_grad.html).

The following experiment uses a separate fresh model so it cannot contaminate the main loop.

In [7]:
def demonstrate_zero_grad():
    model = make_model()
    optimizer = torch.optim.SGD(model.parameters(), lr=0.2)
    loss_fn = nn.CrossEntropyLoss()
    x, y, _, _ = tiny_data()
    optimizer.zero_grad(set_to_none=True)
    loss_fn(model(x), y).backward()
    first = {name: p.grad.clone() for name, p in model.named_parameters()}
    first_norms = gradient_norms(model)
    # Recompute the forward pass to create a new graph. No optimizer step occurs here.
    loss_fn(model(x), y).backward()
    second_norms = gradient_norms(model)
    for name, parameter in model.named_parameters():
        torch.testing.assert_close(parameter.grad, 2 * first[name])
    optimizer.zero_grad(set_to_none=True)
    cleared = gradient_norms(model)
    assert all(value is None for value in cleared.values())
    print('After one backward:', first_norms)
    print('After two backwards WITHOUT zero_grad:', second_norms)
    print('After zero_grad(set_to_none=True):', cleared)
    return first_norms, second_norms, cleared

first_norms, doubled_norms, cleared_norms = demonstrate_zero_grad()

After one backward: {'weight': 0.4124789834022522, 'bias': 0.0}
After two backwards WITHOUT zero_grad: {'weight': 0.8249579668045044, 'bias': 0.0}
After zero_grad(set_to_none=True): {'weight': None, 'bias': None}


## 7. train() versus eval() versus no_grad()

`model.train()` activates training behavior in layers such as Dropout. `model.eval()` selects evaluation behavior; Dropout stops dropping activations, and BatchNorm normally uses its stored running statistics. A plain linear layer behaves the same in both modes, so a separate Dropout probe makes the difference visible.

**Mode and gradient tracking are separate.** Evaluation mode does not freeze parameters. `torch.no_grad()` disables recording of new differentiable operations; it does not change the mode or clear existing gradients. [PyTorch autograd guidance](https://docs.pytorch.org/docs/main/notes/autograd.html).

With dropout probability 0.5, retained training activations are doubled to preserve their expected value; evaluation returns the original values. We add zero in the probe to ensure an actual operation occurs even when evaluation Dropout returns its input unchanged.

In [8]:
def demonstrate_modes():
    # Dropout makes train/eval behavior visible; the main linear model has no dropout.
    dropout = nn.Dropout(p=0.5)
    x = torch.ones(16, requires_grad=True)
    with torch.random.fork_rng():
        torch.manual_seed(7)
        dropout.train()
        training_output = dropout(x)
        dropout.eval()
        # Add zero so there is a real operation even when eval Dropout is identity.
        evaluation_output = dropout(x) + 0
        with torch.no_grad():
            untracked_output = dropout(x) + 0
    print('train() Dropout output:', training_output.detach().tolist())
    print('eval() Dropout output:', evaluation_output.detach().tolist())
    print('eval() still tracks gradients:', evaluation_output.requires_grad)
    print('eval() + no_grad tracks gradients:', untracked_output.requires_grad)
    return training_output, evaluation_output, untracked_output

train_output, eval_output, untracked_output = demonstrate_modes()
assert eval_output.requires_grad and not untracked_output.requires_grad

train() Dropout output: [2.0, 2.0, 0.0, 0.0, 0.0, 0.0, 2.0, 0.0, 0.0, 0.0, 2.0, 2.0, 2.0, 2.0, 2.0, 0.0]
eval() Dropout output: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
eval() still tracks gradients: True
eval() + no_grad tracks gradients: False


## 8. Batch, epoch, and optimizer step

A **batch** is a group processed together. An **epoch** is one pass through all training examples. Here, a **step** is one optimizer update. Without gradient accumulation and without dropping the last batch:

$$\text{steps per epoch}=\lceil N/B\rceil.$$

Four examples with batch size two give two steps per epoch. Twenty epochs give forty optimizer steps. A batch size of three gives a final batch of one, still two steps per epoch.

Read the function below line by line, then run it. It starts a **fresh model**, separate from the one-step and accumulation experiments. A local shuffle seed makes batch order reproducible. CPU float32 is used; bitwise reproducibility across different PyTorch versions/hardware is not promised.

The main loop follows `zero_grad → forward → loss → backward → inspect gradients → step`. Validation is measured after the update with evaluation mode and no gradient tracking. It never triggers an update. [PyTorch optimization tutorial](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html).

In [9]:
def train(epochs=20, batch_size=2, learning_rate=0.2, verbose=True):
    if isinstance(epochs, bool) or not isinstance(epochs, int) or epochs < 1:
        raise ValueError('epochs must be a positive integer')
    if isinstance(batch_size, bool) or not isinstance(batch_size, int) or batch_size < 1:
        raise ValueError('batch_size must be a positive integer')
    if not math.isfinite(learning_rate) or learning_rate <= 0:
        raise ValueError('learning_rate must be positive and finite')
    x_train, y_train, x_valid, y_valid = tiny_data()
    model = make_model()
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
    loss_fn = nn.CrossEntropyLoss()  # Give it raw logits, not softmax probabilities.
    generator = torch.Generator().manual_seed(7)
    history = []
    step = 0
    model.eval()
    with torch.no_grad():
        initial_train = loss_fn(model(x_train), y_train).item()
        initial_valid = loss_fn(model(x_valid), y_valid).item()
    history.append(dict(step=0, epoch=0, batch_size=0, loss_before=None, loss_after=None,
                        weight_grad_norm=None, bias_grad_norm=None,
                        train_loss=initial_train, validation_loss=initial_valid))

    for epoch in range(1, epochs + 1):
        indices = torch.randperm(len(x_train), generator=generator)
        for start in range(0, len(x_train), batch_size):
            batch = indices[start:start + batch_size]
            x_batch, y_batch = x_train[batch], y_train[batch]
            model.train()                          # Set training behavior.
            optimizer.zero_grad(set_to_none=True)  # Clear the PREVIOUS gradients.
            logits = model(x_batch)                # Forward pass: [batch, 2] logits.
            loss = loss_fn(logits, y_batch)        # Mean cross-entropy for this batch.
            loss_before = loss.item()
            loss.backward()                       # Backpropagation computes gradients.
            norms = gradient_norms(model)          # Inspect gradients BEFORE updating.
            optimizer.step()                      # SGD updates parameters using gradients.
            with torch.no_grad():
                loss_after = loss_fn(model(x_batch), y_batch).item()  # SAME batch, new forward.
            model.eval()                           # Evaluation behavior, separate from no_grad.
            with torch.no_grad():
                train_loss = loss_fn(model(x_train), y_train).item()
                valid_loss = loss_fn(model(x_valid), y_valid).item()
            step += 1
            history.append(dict(step=step, epoch=epoch, batch_size=len(batch),
                                loss_before=loss_before, loss_after=loss_after,
                                weight_grad_norm=norms['weight'], bias_grad_norm=norms['bias'],
                                train_loss=train_loss, validation_loss=valid_loss))
            if verbose:
                print(f'step={step:02d} epoch={epoch:02d} batch={len(batch)} '
                      f'loss {loss_before:.6f} -> {loss_after:.6f} '
                      f'grad norms: weight={norms["weight"]:.6f}, bias={norms["bias"]:.6f} '
                      f'validation={valid_loss:.6f}')
    return model, history

In [10]:
trained_model, history = train(epochs=20, batch_size=2, learning_rate=0.2)
assert len(history) == 41  # Initial measurement plus forty optimizer steps.
print("Full training loss:", history[0]["train_loss"], "->", history[-1]["train_loss"])
print("Held-out validation loss:", history[0]["validation_loss"], "->", history[-1]["validation_loss"])

step=01 epoch=01 batch=2 loss 0.693147 -> 0.559341 grad norms: weight=0.471405, bias=0.707107 validation=0.666584
step=02 epoch=01 batch=2 loss 0.762096 -> 0.628998 grad norms: weight=0.375782, bias=0.754172 validation=0.635322
step=03 epoch=02 batch=2 loss 0.635388 -> 0.613914 grad norms: weight=0.329288, bias=0.018754 validation=0.614075
step=04 epoch=02 batch=2 loss 0.589674 -> 0.557021 grad norms: weight=0.407496, bias=0.035910 validation=0.588485
step=05 epoch=03 batch=2 loss 0.524183 -> 0.480074 grad norms: weight=0.477386, bias=0.019293 validation=0.559788
step=06 epoch=03 batch=2 loss 0.600757 -> 0.591731 grad norms: weight=0.212873, bias=0.008273 validation=0.547506
step=07 epoch=04 batch=2 loss 0.591731 -> 0.582904 grad norms: weight=0.210532, bias=0.007438 validation=0.535626
step=08 epoch=04 batch=2 loss 0.443693 -> 0.409890 grad norms: weight=0.416540, bias=0.031805 validation=0.512540
step=09 epoch=05 batch=2 loss 0.499978 -> 0.428210 grad norms: weight=0.269925, bias=0.5

## 9. Save the measured table

`loss_before` and `loss_after` refer to the **same current training batch** on either side of its update. They are blank at step zero, when no update has happened. `train_loss` and `validation_loss` are evaluated on their fixed complete sets after each update.

Do not compare different batches as though they contained identical examples: one batch can be harder than the next. The full-set columns make the overall trend easier to inspect. The CSV and Markdown table contain actual measured values, not illustrative values. Rerunning this cell replaces files in the notebook output folder; the bundled CLI reference table is kept separately.

In [11]:
def history_table(history):
    columns = list(history[0])
    lines = ['| ' + ' | '.join(columns) + ' |', '| ' + ' | '.join(['---'] * len(columns)) + ' |']
    for row in history:
        values = ['—' if row[key] is None else f'{row[key]:.6f}' if isinstance(row[key], float)
                  else str(row[key]) for key in columns]
        lines.append('| ' + ' | '.join(values) + ' |')
    return '\n'.join(lines) + '\n'

def save_history(history, output):
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    with (output / 'loss_by_step.csv').open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(history[0]))
        writer.writeheader()
        writer.writerows(history)
    (output / 'loss_by_step.md').write_text('# Measured loss by optimizer step\n\n' + history_table(history), encoding='utf-8')
    print('Saved:', output / 'loss_by_step.csv', 'and', output / 'loss_by_step.md')

save_history(history, OUTPUT)
display(Markdown(history_table(history)))

Saved: /Users/smohandoss/Documents/Codex/2026-09-09/files-mentioned-by-the-user-recipetriage/outputs/recipetriage/training-results/notebook/loss_by_step.csv and /Users/smohandoss/Documents/Codex/2026-09-09/files-mentioned-by-the-user-recipetriage/outputs/recipetriage/training-results/notebook/loss_by_step.md


| step | epoch | batch_size | loss_before | loss_after | weight_grad_norm | bias_grad_norm | train_loss | validation_loss |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 0 | 0 | 0 | — | — | — | — | 0.693147 | 0.693147 |
| 1 | 1 | 2 | 0.693147 | 0.559341 | 0.471405 | 0.707107 | 0.660718 | 0.666584 |
| 2 | 1 | 2 | 0.762096 | 0.628998 | 0.375782 | 0.754172 | 0.626199 | 0.635322 |
| 3 | 2 | 2 | 0.635388 | 0.613914 | 0.329288 | 0.018754 | 0.601794 | 0.614075 |
| 4 | 2 | 2 | 0.589674 | 0.557021 | 0.407496 | 0.035910 | 0.572744 | 0.588485 |
| 5 | 3 | 2 | 0.524183 | 0.480074 | 0.477386 | 0.019293 | 0.540416 | 0.559788 |
| 6 | 3 | 2 | 0.600757 | 0.591731 | 0.212873 | 0.008273 | 0.526606 | 0.547506 |
| 7 | 4 | 2 | 0.591731 | 0.582904 | 0.210532 | 0.007438 | 0.513299 | 0.535626 |
| 8 | 4 | 2 | 0.443693 | 0.409890 | 0.416540 | 0.031805 | 0.487845 | 0.512540 |
| 9 | 5 | 2 | 0.499978 | 0.428210 | 0.269925 | 0.555196 | 0.473399 | 0.496648 |
| 10 | 5 | 2 | 0.518588 | 0.436646 | 0.343942 | 0.566249 | 0.452809 | 0.480703 |
| 11 | 6 | 2 | 0.541184 | 0.533450 | 0.197004 | 0.008616 | 0.442157 | 0.470984 |
| 12 | 6 | 2 | 0.350864 | 0.327939 | 0.340458 | 0.044708 | 0.424166 | 0.454071 |
| 13 | 7 | 2 | 0.395346 | 0.378946 | 0.268749 | 0.105691 | 0.410363 | 0.440453 |
| 14 | 7 | 2 | 0.441780 | 0.429295 | 0.236083 | 0.086323 | 0.399061 | 0.430530 |
| 15 | 8 | 2 | 0.296716 | 0.279449 | 0.293930 | 0.046696 | 0.385086 | 0.417008 |
| 16 | 8 | 2 | 0.490722 | 0.484015 | 0.182755 | 0.018104 | 0.376803 | 0.409333 |
| 17 | 9 | 2 | 0.269592 | 0.254951 | 0.269657 | 0.047741 | 0.364733 | 0.397446 |
| 18 | 9 | 2 | 0.474515 | 0.468129 | 0.178022 | 0.020707 | 0.357143 | 0.390392 |
| 19 | 10 | 2 | 0.324768 | 0.312621 | 0.214590 | 0.125270 | 0.347907 | 0.380346 |
| 20 | 10 | 2 | 0.383192 | 0.372677 | 0.204038 | 0.108143 | 0.339817 | 0.373604 |
| 21 | 11 | 2 | 0.329301 | 0.292839 | 0.207298 | 0.385828 | 0.333909 | 0.372001 |
| 22 | 11 | 2 | 0.374979 | 0.330534 | 0.204827 | 0.438401 | 0.323595 | 0.357364 |
| 23 | 12 | 2 | 0.316656 | 0.282939 | 0.196307 | 0.372185 | 0.318137 | 0.356194 |
| 24 | 12 | 2 | 0.353336 | 0.313151 | 0.193619 | 0.416959 | 0.309083 | 0.342712 |
| 25 | 13 | 2 | 0.190979 | 0.182993 | 0.196552 | 0.045552 | 0.302030 | 0.335269 |
| 26 | 13 | 2 | 0.421067 | 0.415657 | 0.161812 | 0.032396 | 0.296457 | 0.330147 |
| 27 | 14 | 2 | 0.331404 | 0.322549 | 0.176445 | 0.117715 | 0.290766 | 0.325933 |
| 28 | 14 | 2 | 0.258983 | 0.249864 | 0.161253 | 0.142946 | 0.285087 | 0.318601 |
| 29 | 15 | 2 | 0.165070 | 0.158917 | 0.171179 | 0.044729 | 0.279488 | 0.312490 |
| 30 | 15 | 2 | 0.400059 | 0.395031 | 0.155215 | 0.035041 | 0.274599 | 0.308051 |
| 31 | 16 | 2 | 0.395031 | 0.390139 | 0.153662 | 0.031793 | 0.269873 | 0.303740 |
| 32 | 16 | 2 | 0.149607 | 0.144440 | 0.155350 | 0.046092 | 0.265079 | 0.298357 |
| 33 | 17 | 2 | 0.300032 | 0.291987 | 0.158580 | 0.125695 | 0.260600 | 0.295526 |
| 34 | 17 | 2 | 0.229214 | 0.221318 | 0.138557 | 0.145070 | 0.256285 | 0.289278 |
| 35 | 18 | 2 | 0.135593 | 0.131276 | 0.141638 | 0.043049 | 0.252196 | 0.284584 |
| 36 | 18 | 2 | 0.373116 | 0.368572 | 0.146561 | 0.037545 | 0.248095 | 0.280952 |
| 37 | 19 | 2 | 0.247776 | 0.226129 | 0.137329 | 0.306297 | 0.244628 | 0.273282 |
| 38 | 19 | 2 | 0.263127 | 0.240177 | 0.147971 | 0.312279 | 0.240301 | 0.273158 |
| 39 | 20 | 2 | 0.360327 | 0.356077 | 0.142437 | 0.033297 | 0.236553 | 0.269836 |
| 40 | 20 | 2 | 0.117029 | 0.113704 | 0.122230 | 0.043750 | 0.233313 | 0.265931 |


## 10. Overfitting and your checkpoint

**Overfitting** means fitting peculiarities or noise in training examples that do not generalize. A common sign is training loss continuing downward while held-out validation loss rises. A lower training loss alone does not establish a better real-world model. A tiny validation set is noisy evidence.

This easy four-parameter example usually improves on both sets. That is **not** an overfitting demonstration or proof that overfitting cannot happen. More epochs do not inevitably cause it. In a larger noisy task, compare train/validation curves, control model capacity, and consider early stopping based on validation; reserve an untouched test set for the final assessment.

Try learning rates 0.02 and 2.0, or batch sizes 1 and 4, each with a fresh call to `train`. Predict changes in the number of updates and loss behavior first. Keep those experiments separate from the saved reference run.

Explain these in your own words before we move on:

1. Which line calculates predictions, which calculates gradients, and which changes weights?
2. Why do gradients double in the accumulation experiment? Does `zero_grad` reset the model?
3. Why must we run a new forward pass to measure loss after `optimizer.step()`?
4. With four examples, batch size two, and twenty epochs, how many optimizer steps occur? Does `eval()` disable gradients?
5. What would falling training loss but rising validation loss suggest?

**Stop here.** SFTTrainer is deliberately deferred until you can explain this loop.